# PREDICTING SCRAP RATIO FOR PRODUCTION PLANNING

## 1. Introduction

Business question:
Can we estimate production scrap before production starts?

This notebook uses a synthetically generated production dataset from the plastic industry, designed to reflect real-world process conditions.

If scrap can be estimated in advance, production planning, material handling and quality control (in some cases, manpower planning) become more reliable and predictable. 

The goal of this study is to predict scrap ratio using only information available before production, such as:

- Product type
- Material properties
- Process parameters
- Planned production quantity

## 2. Load Data & Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
import lightgbm as lgb

import warnings
warnings.simplefilter("ignore")

# Load data
df = pd.read_parquet("production_data_cleaned.parquet")

print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)

Shape: (116958, 12)

Dtypes:
base_code              category
embossing              category
gloss                  category
printed                    bool
date             datetime64[ns]
thickness               float64
colorname                object
material_type          category
jumbo                      bool
recycle                    bool
total_kg                float64
scrap_ratio             float64
dtype: object


## 3. Data Preparation

Time-based ordering is used to simulate real production flow.

In [2]:
df_model = df.copy().sort_values("date")
y = df_model["scrap_ratio"]
X = df_model.drop(columns=["scrap_ratio", "date"])
print("Data is sorted by date so the model learns from past records and is tested on future records.")

Data is sorted by date so the model learns from past records and is tested on future records.


## 4. Train / Validation / Test Split

Data is split chronologically to prevent leakage.

In [3]:
n = len(df_model)

train_end = int(n * 0.7)
val_end = int(n * 0.85)
X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()
X_val = X.iloc[train_end:val_end].copy()
y_val = y.iloc[train_end:val_end].copy()
X_test = X.iloc[val_end:].copy()
y_test = y.iloc[val_end:].copy()

print("Train set      :", X_train.shape)
print("Validation set :", X_val.shape)
print("Test set       :", X_test.shape)

Train set      : (81870, 10)
Validation set : (17544, 10)
Test set       : (17544, 10)


## 5. Feature Overview

Categorical features are formatted to ensure optimal performance across different models.

In [4]:
cat_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

for col in cat_cols:
    X_train[col] = X_train[col].astype(str)
    X_val[col] = X_val[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print("\nCategorical columns used by the models:")
print(cat_cols)


Categorical columns used by the models:
['base_code', 'embossing', 'gloss', 'printed', 'colorname', 'material_type', 'jumbo', 'recycle']


## 6. Encoding Strategy

* Categorical features are encoded using leakage-safe K-Fold Target Encoding and Frequency Encoding.

* Target Encoding captures the historical scrap behavior of each category, while K-Fold splitting prevents the model from seeing its own target value during encoding.

* Smoothing controls how much category averages are shrunk toward the global mean.A value of 20 is used as a reasonable default to balance; learning meaningful patterns from frequent categories and preventing overfitting for rare categories.

In [5]:
class KFoldTargetEncoder:
    def __init__(self, cols, n_splits=5, smoothing=20, random_state=42):
        self.cols = cols
        self.n_splits = n_splits
        self.smoothing = smoothing
        self.random_state = random_state
        self.global_mean = None
        self.maps = {}

    def fit_transform(self, X, y):
        X_out = X.copy()
        self.global_mean = y.mean()

        kf = KFold(
            n_splits=self.n_splits,
            shuffle=True,
            random_state=self.random_state
        )

        for col in self.cols:
            encoded = pd.Series(index=X.index, dtype=float)

            for train_idx, valid_idx in kf.split(X):
                X_tr = X.iloc[train_idx]
                y_tr = y.iloc[train_idx]
                X_val_fold = X.iloc[valid_idx]

                stats = y_tr.groupby(X_tr[col]).agg(["mean", "count"])

                smooth = (
                    (stats["mean"] * stats["count"] + self.global_mean * self.smoothing)
                    / (stats["count"] + self.smoothing)
                )

                encoded.iloc[valid_idx] = X_val_fold[col].map(smooth)

            X_out[f"{col}_te"] = encoded.fillna(self.global_mean)

            full_stats = y.groupby(X[col]).agg(["mean", "count"])
            self.maps[col] = (
                (full_stats["mean"] * full_stats["count"] + self.global_mean * self.smoothing)
                / (full_stats["count"] + self.smoothing)
            )

        return X_out

    def transform(self, X):
        X_out = X.copy()

        for col in self.cols:
            X_out[f"{col}_te"] = X_out[col].map(self.maps[col]).fillna(self.global_mean)

        return X_out


target_encoder = KFoldTargetEncoder(
    cols=cat_cols,
    n_splits=5,
    smoothing=20,
    random_state=42
)

X_train_enc = target_encoder.fit_transform(X_train, y_train)
X_val_enc = target_encoder.transform(X_val)
X_test_enc = target_encoder.transform(X_test)

for col in cat_cols:
    freq_map = X_train[col].value_counts(normalize=True)

    X_train_enc[f"{col}_freq"] = X_train[col].map(freq_map)
    X_val_enc[f"{col}_freq"] = X_val[col].map(freq_map).fillna(0)
    X_test_enc[f"{col}_freq"] = X_test[col].map(freq_map).fillna(0)

X_train_enc = X_train_enc.drop(columns=cat_cols)
X_val_enc = X_val_enc.drop(columns=cat_cols)
X_test_enc = X_test_enc.drop(columns=cat_cols)

print("Encoding completed.")
print("Encoded train shape:", X_train_enc.shape)

Encoding completed.
Encoded train shape: (81870, 18)


## 7. Model Training

To capture different types of patterns in the data, three models are used:

* CatBoost → strong handling of categorical production data
* XGBoost → captures complex non-linear relationships
* LightGBM → fast and efficient boosting model

These models are combined using a validation-based ensemble, allowing the final prediction to benefit from the strengths of each model.

In [6]:
%%time

print("\n==============================")
print("MODEL TRAINING: CATBOOST")
print("==============================")

cat_model = CatBoostRegressor(
    iterations=2000,
    depth=6,
    learning_rate=0.05,
    loss_function="RMSE",
    random_seed=42,
    verbose=200
)

cat_model.fit(
    X_train,
    y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    use_best_model=True
)

print("\nCatBoost training completed.")
print("Best iteration:", cat_model.get_best_iteration())
print("Validation RMSE:", round(cat_model.get_best_score()["validation"]["RMSE"], 5))


print("\n==============================")
print("MODEL TRAINING: XGBOOST")
print("==============================")

xgb_model = XGBRegressor(
    n_estimators=2000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    verbose=200
)

print("\nXGBoost training completed.")


print("\n==============================")
print("MODEL TRAINING: LIGHTGBM")
print("==============================")

lgb_model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.0,
    reg_lambda=0.1,
    objective="regression",
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train_enc,
    y_train,
    eval_set=[(X_val_enc, y_val)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100),
        lgb.log_evaluation(period=200)
    ]
)

print("\nLightGBM training completed.")
print("Best iteration:", lgb_model.best_iteration_)


MODEL TRAINING: CATBOOST
0:	learn: 0.1426405	test: 0.1394993	best: 0.1394993 (0)	total: 71.3ms	remaining: 2m 22s
200:	learn: 0.0455225	test: 0.0600643	best: 0.0600643 (200)	total: 2.12s	remaining: 19s
400:	learn: 0.0436642	test: 0.0588005	best: 0.0588005 (400)	total: 3.9s	remaining: 15.6s
600:	learn: 0.0426750	test: 0.0581857	best: 0.0581857 (600)	total: 5.61s	remaining: 13.1s
800:	learn: 0.0420255	test: 0.0578810	best: 0.0578810 (799)	total: 7.38s	remaining: 11s
1000:	learn: 0.0415581	test: 0.0577226	best: 0.0577200 (997)	total: 9.05s	remaining: 9.03s
1200:	learn: 0.0411423	test: 0.0575243	best: 0.0575243 (1200)	total: 10.9s	remaining: 7.25s
1400:	learn: 0.0407733	test: 0.0573947	best: 0.0573935 (1399)	total: 12.7s	remaining: 5.41s
1600:	learn: 0.0404669	test: 0.0572706	best: 0.0572706 (1600)	total: 14.4s	remaining: 3.6s
1800:	learn: 0.0401995	test: 0.0571811	best: 0.0571811 (1800)	total: 16.1s	remaining: 1.78s
1999:	learn: 0.0399781	test: 0.0571158	best: 0.0571111 (1983)	total: 17.8

## 8. Ensemble

The ensemble weights are optimized on the validation set to find the best blend of CatBoost, XGBoost, and LightGBM. 

In [7]:
print("\n============================")
print(" VALIDATION ENSEMBLE SEARCH")
print("============================")

cat_val_pred = np.clip(cat_model.predict(X_val), 0, 1)
xgb_val_pred = np.clip(xgb_model.predict(X_val_enc), 0, 1)
lgb_val_pred = np.clip(lgb_model.predict(X_val_enc), 0, 1)

best_score = np.inf
best_weights = None

for w_cat in np.arange(0, 1.05, 0.05):
    for w_xgb in np.arange(0, 1.05 - w_cat, 0.05):
        w_lgb = 1 - w_cat - w_xgb

        val_pred = (
            w_cat * cat_val_pred +
            w_xgb * xgb_val_pred +
            w_lgb * lgb_val_pred
        )

        val_pred = np.clip(val_pred, 0, 1)
        rmse = np.sqrt(mean_squared_error(y_val, val_pred))

        if rmse < best_score:
            best_score = rmse
            best_weights = (w_cat, w_xgb, w_lgb)

print("Best validation weights:")
print("CatBoost:", best_weights[0])
print("XGBoost :", best_weights[1])
print("LightGBM:", best_weights[2])
print("Validation RMSE:", round(best_score, 5))


 VALIDATION ENSEMBLE SEARCH
Best validation weights:
CatBoost: 0.25
XGBoost : 0.30000000000000004
LightGBM: 0.44999999999999996
Validation RMSE: 0.0564


## 9. Final Evaulation

Results below show how well the model can predict scrap for new production orders.

Each model is also evaluated individually to see whether the ensemble actually improves performance.

In [8]:
cat_test_pred = np.clip(cat_model.predict(X_test), 0, 1)
xgb_test_pred = np.clip(xgb_model.predict(X_test_enc), 0, 1)
lgb_test_pred = np.clip(lgb_model.predict(X_test_enc), 0, 1)

w_cat, w_xgb, w_lgb = best_weights

final_pred = (
    w_cat * cat_test_pred +
    w_xgb * xgb_test_pred +
    w_lgb * lgb_test_pred
)

final_pred = np.clip(final_pred, 0, 1)

print("\n====================")
print(" FINAL TEST RESULTS")
print("====================")

mse = mean_squared_error(y_test, final_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, final_pred)
r2 = r2_score(y_test, final_pred)

print("MSE :", mse)
print("RMSE:", rmse)
print("MAE :", mae)
print("R2  :", r2)


def evaluate_model(name, y_true, pred):
    print(f"\n{name}")
    print("-" * len(name))
    print("RMSE:", np.sqrt(mean_squared_error(y_true, pred)))
    print("MAE :", mean_absolute_error(y_true, pred))
    print("R2  :", r2_score(y_true, pred))


print("\n====================")
print(" MODEL COMPARISON")
print("====================")

evaluate_model("CatBoost", y_test, cat_test_pred)
evaluate_model("XGBoost", y_test, xgb_test_pred)
evaluate_model("LightGBM", y_test, lgb_test_pred)


 FINAL TEST RESULTS
MSE : 0.0036558799886803398
RMSE: 0.06046387341777187
MAE : 0.04077814076375133
R2  : 0.8243507086357608

 MODEL COMPARISON

CatBoost
--------
RMSE: 0.060232222760437784
MAE : 0.04073214475796516
R2  : 0.8256940340511931

XGBoost
-------
RMSE: 0.061677309340881926
MAE : 0.04169925880984208
R2  : 0.8172298323552383

LightGBM
--------
RMSE: 0.06080992871864407
MAE : 0.04098714006509233
R2  : 0.8223343537538607


## 10. Conclusion

Three different models were trained and combined to leverage their individual strengths.

CatBoost achieved the best standalone performance.  
XGBoost and LightGBM showed similar behavior, capturing additional patterns in the data.

The models were then combined using a validation-based ensemble.  
The final ensemble reached an R² score of 0.82 on the test set.

## 11. Where This Approach Can Be Applied

This approach can be integrated into production planning workflows where scrap variability impacts cost and efficiency.

Typical use cases include:

- **Pre-production planning**  
  Estimating expected scrap ratio for each order before production starts

- **Material requirement planning**  
  Adjusting raw material allocation based on predicted scrap levels

- **Production risk identification**  
  Detecting orders with higher scrap risk in advance

- **Process optimization**  
  Analyzing which product or process conditions lead to higher scrap

- **Scenario analysis**  
  Comparing different production setups and their expected scrap impact